# Clean Large Steam Plant Cost Data
*Steven Dahlke*


## Overview
This notebook documents and reports on the data cleaning effort for FERC Form 1 cost data for large steam plants. Data is sourced from Catalyst Cooperative's Public Utility Data Liberation Project (PUDL) data files as detailed throughout this notebook.

## Prepare FERC Form 1 Large Plant Cost Data for Cleaning
Historic annual capital expenditure (CapEx) and operating expenditure (OpEx) data were downloaded via PUDL to CSV form and stored locally (https://data.catalyst.coop/pudl/core_ferc1__yearly_steam_plants_sched402).
This table contains generating plant statistics for steam plants with a capacity of 25+ MW, internal combustion and gas-turbine plants of 10+ MW, and all nuclear plants. As reported in Schedule 402 of FERC Form 1 and extracted from the f1_gnrt_plant table in FERC's Visual FoxPro Database. 

First, the software libraries are installed, then the project folder is defined as one directory above the current working directory (where this notebook file is located). Then the large plants data is read in locally.

In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
# Define project directory
root = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(f"Project root: {root}")

Project root: C:\Users\sjdahlke\Documents\2025 Plant Costs


For this analysis, we will consider the recent cross-sectional dataset consisting of 2023 reported costs across the plants. Cleaning data across a broader history would be a singificantly more difficult undertaking given the existing state of the Form 1 dataset.

In [3]:
path = os.path.join(root, 'data', 'core_ferc1__yearly_steam_plants_sched402.csv')
df = pd.read_csv(path)
df = df[df["report_year"]==2023]
df.head(3)

,rowid,record_id,utility_id_ferc1,report_year,plant_name_ferc1,plant_type,construction_type,construction_year,installation_year,capacity_mw,...,opex_rents,opex_allowances,opex_engineering,opex_structures,opex_boiler,opex_plants,opex_misc_steam,opex_production_total,opex_per_mwh,asset_retirement_cost
63,31687,steam_electric_generating_plant_statistics_lar...,206,2023,*dolet hills (3),steam,outdoor,1986.0,1986.0,289.97,...,NaN,NaN,NaN,-13181.0,8118.0,507.0,134421.0,2769751.0,NaN,NaN
90,31695,steam_electric_generating_plant_statistics_lar...,206,2023,*flint creek (1),steam,outdoor,1978.0,1978.0,279.00,...,NaN,85088.0,511676.0,819742.0,2926162.0,386663.0,1255625.0,40253934.0,37.9,12402374.0
140,31697,steam_electric_generating_plant_statistics_lar...,206,2023,*pirkey (2),steam,outdoor,1985.0,1985.0,619.34,...,NaN,24777.0,33537.0,35546.0,1043990.0,165098.0,741318.0,55814930.0,75.2,NaN


In [4]:
# Display all the column names
df.columns

Index(['rowid', 'record_id', 'utility_id_ferc1', 'report_year',
       'plant_name_ferc1', 'plant_type', 'construction_type',
       'construction_year', 'installation_year', 'capacity_mw',
       'peak_demand_mw', 'plant_hours_connected_while_generating',
       'plant_capability_mw', 'water_limited_capacity_mw',
       'not_water_limited_capacity_mw', 'avg_num_employees',
       'net_generation_mwh', 'capex_land', 'capex_structures',
       'capex_equipment', 'capex_total', 'capex_per_mw', 'opex_operations',
       'opex_fuel', 'opex_coolants', 'opex_steam', 'opex_steam_other',
       'opex_transfer', 'opex_electric', 'opex_misc_power', 'opex_rents',
       'opex_allowances', 'opex_engineering', 'opex_structures', 'opex_boiler',
       'opex_plants', 'opex_misc_steam', 'opex_production_total',
       'opex_per_mwh', 'asset_retirement_cost'],
      dtype='object')

Next, the utility names associated with *utility_id_ferc1* are included, which will be helpful in understanding this data when we clean up the plant entries. We're going to bring in for merging *utility_id_ferc1*, *utility_id_ferc1_label*, and *report_year* from the PUDL FERC1 table with utility balance sheets: (https://data.catalyst.coop/pudl/core_ferc1__yearly_balance_sheet_assets_sched110).


In [5]:
path = os.path.join(root, 'data', 'core_ferc1__yearly_balance_sheet_assets_sched110.csv')
columns_to_read = ['utility_id_ferc1', 'utility_id_ferc1_label', 'report_year']
util = pd.read_csv(path, usecols=columns_to_read)
util = util.drop_duplicates(subset=['utility_id_ferc1', 'utility_id_ferc1_label', 'report_year'])
df = df.merge(util, on=['utility_id_ferc1', 'report_year'], how='left')

In [6]:
columns = df.columns.tolist()
columns.remove('utility_id_ferc1_label')
columns.insert(3, 'utility_id_ferc1_label')
df = df[columns]

In [7]:
# For cleaning, it will work to sort the dataset by utility (the reporting entity), then plant name, then year
df = df.sort_values(by=["utility_id_ferc1_label", "plant_name_ferc1", "installation_year","capacity_mw", "report_year",])

# Define new columns for EIA names
new_columns = ["plant_id_eia", "plant_name_eia", "generator_id", "units", "data_questionable","retired"]
for i, col in enumerate(new_columns):
    df.insert(loc=6 + i, column=col, value=None)  # Inserts None as default values

df['cleaningNotes'] = ""

The EIA data are used to assign locations and match other characteristics to the FERC dataset. This includes PUDL's core Plants dataset, which has the location of each power plant (https://data.catalyst.coop/pudl/core_eia__entity_plants), and information is linked from the SCD_Generators table which has many other generator-level characteristics (https://data.catalyst.coop/pudl/core_eia860__scd_generators), as well as the EIA utility names (https://data.catalyst.coop/pudl/core_eia__entity_utilities).  

In [8]:
# keep useful data from eia dataset
columns = ['plant_id_eia','generator_id','utility_id_eia','report_date',
          'operational_status','ownership_code','capacity_mw',
          'energy_storage_capacity_mwh','prime_mover_code',
           'energy_source_code_1','fuel_type_code_pudl',
          'planned_generator_retirement_date','carbon_capture',
          'technology_description','generator_retirement_date',
           'owned_by_non_utility']
gen = pd.read_csv(os.path.join(root,'data','core_eia860__scd_generators.csv'), low_memory=False)

#get utility names to link to the utility_id_eia from another PUDL table
util = pd.read_csv(os.path.join(root,'data','core_eia__entity_utilities.csv'))
gen = gen.merge(util, on=['utility_id_eia'], how='left')
columns = gen.columns.tolist()
columns.remove('utility_name_eia')
columns.insert(3, 'utility_name_eia')
gen = gen[columns]

# Merge in the plant name too
plant = pd.read_csv(os.path.join(root,'data','core_eia__entity_plants.csv'))
gen = gen.merge(plant, on=['plant_id_eia'], how='left')
columns = gen.columns.tolist()
columns.remove('plant_name_eia')
columns.insert(1, 'plant_name_eia')
gen = gen[columns]

gen = gen.sort_values(by=["utility_name_eia", "plant_name_eia"])
gen.to_csv(os.path.join(root,'data','eia_gen.csv'), index=False)

In [9]:
#Define functions that'll help with cleaning
def delete_plant_rows(plant_names: list) -> pd.DataFrame:
    global df
    df = df[~df["plant_name_ferc1"].isin(plant_names)].reset_index(drop=True)
    return df

def add_eia_info(plant_name_ferc1: str, plant_id_eia: int, plant_name_eia: str, generator_id: str) -> pd.DataFrame:
    global df
    df.loc[df["plant_name_ferc1"] == plant_name_ferc1, ["plant_id_eia", "plant_name_eia", "generator_id"]] = [plant_id_eia, plant_name_eia, generator_id]
    return df

def add_eia_info2(plant_name_ferc1: str, plant_id_eia: int, generator_id: str, units: int = None) -> pd.DataFrame:
    global df
    df.loc[df["plant_name_ferc1"] == plant_name_ferc1, ["plant_id_eia", "generator_id", "units"]] = [plant_id_eia, generator_id, units]
    return df
    
def add_notes(plant_name_ferc1: str, notes: str) -> pd.DataFrame:
    global df
    df.loc[df["plant_name_ferc1"] == plant_name_ferc1, "cleaningNotes"] = notes
    return df

def data_questionable(plant_name_ferc1: str = None):
    global df
    df.loc[df['plant_name_ferc1'] == plant_name_ferc1, 'data_questionable'] = 1
    return df

def change_plant_name(utility_name: str, plant_name_ferc1: str, new_name: str) -> pd.DataFrame:
    global df
    mask = (
        (df["utility_id_ferc1_label"] == utility_name) &
        (df["plant_name_ferc1"] == plant_name_ferc1)
    )
    df.loc[mask, "plant_name_ferc1"] = new_name
    return df

def is_retired(plant_name_ferc1: str) -> pd.DataFrame:
    global df
    df.loc[df["plant_name_ferc1"] == plant_name_ferc1, "retired"] = 1
    return df

## Clean Power Plants Data

In many cases the power plant in Form 1 is made up of multiple generating units. Only 1 unit is assigned that matches the fuel technology of the plant reported in Form 1, to facilitate proper matching with EIA technology data. The fuel type is not available in Form 1 so that has to be assigned via EIA information.

In [10]:
# Delete all rows with plant_name_ferc1 = "total" as these are company totals for the reporting year across all plants 
# and we are just keeping plant level data
df = delete_plant_rows(['total'])

### AEP Generating Company

In [11]:
# Rockport
# Two units, and values are split in half between unit-level entries for AEP Generating Company and Indiana Michigan Power
df = delete_plant_rows(['rockport total aeg', 'rockport total plant', 'rockport total i&m'])
df = add_eia_info("rockport unit 1 aeg", 6166, "Rockport", 1)
df = add_eia_info("rockport unit 1 i&m", 6166, "Rockport", 1)
df = add_eia_info("rockport unit 2 aeg", 6166, "Rockport", 2)
df = add_eia_info("rockport unit 2 i&m", 6166, "Rockport", 2)
df = add_notes('rockport unit 1 aeg', 'Plant has two units, and values are split in half for AEP and IN MIPower')

### Alabama Power Company

In [12]:
# barry
# barry cc
df = add_eia_info(plant_name_ferc1="barry", plant_id_eia=3, plant_name_eia="Barry", generator_id=5)
df = add_notes("barry", "unit 5 is largest coal unit at 789MW, unit 4 is 404MW coal, units 1& 2 are each 153 MW natural gas steam units.")
df = add_eia_info(plant_name_ferc1='barry cc', plant_id_eia=3, plant_name_eia='Barry', generator_id='A1CT')
df = add_notes('barry cc', '8 natural gas combined cycle units')

In [13]:
# calhoun
df = add_notes('calhoun', '4 CT units')
df = add_eia_info(plant_name_ferc1='calhoun', plant_id_eia=55409, plant_name_eia='Calhoun Power Co I LLC', generator_id='CAL1')

In [14]:
# central alabama
df = add_notes('central alabama', '4 unit CC')
df = add_eia_info(plant_name_ferc1='central alabama', plant_id_eia=55440, plant_name_eia='Tenaska Central Alabama', generator_id='CTG1')

In [15]:
#e. c. gaston unit 5, ernest c. gaston, ernest c. gaston - ct
df = add_eia_info(plant_name_ferc1='e. c. gaston unit 5', plant_id_eia=26, plant_name_eia='E C Gaston', generator_id='5')
df = add_notes(plant_name_ferc1='ernest c. gaston', notes='4 unit NG steam turbine')
df = add_eia_info(plant_name_ferc1='ernest c. gaston', plant_id_eia=26, plant_name_eia='E C Gaston', generator_id='1')
df = add_eia_info(plant_name_ferc1='ernest c. gaston - ct', plant_id_eia=26, plant_name_eia='E C Gaston', generator_id='GT4')

In [16]:
p = "gadsden"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7, generator_id='1', units=2)
df = is_retired(plant_name_ferc1=p)


In [17]:
# greene county - respondent's portion, greene county ct, greene county-total, greene county
df = add_notes(plant_name_ferc1='greene county-total', notes='this a 2-unit NG steam plant, unit 1 is 299MW, unit 2 is 269MW')
df = add_eia_info(plant_name_ferc1='greene county-total', plant_id_eia=10, plant_name_eia='Greene County', generator_id='1')

df = add_notes(plant_name_ferc1='greene county ct', notes='9 80 MW CT units')
df = add_eia_info(plant_name_ferc1='greene county ct', plant_id_eia=10, plant_name_eia='Greene County', generator_id='GT2')

df = delete_plant_rows(plant_names=["greene county - respondent's portion", "greene county"])

In [18]:
# j h miller - respondent's portion, j h miller - total
df = delete_plant_rows(plant_names=["j h miller - respondent's portion"])
df = add_notes(plant_name_ferc1="j h miller - total", notes='4 unit coal plant')
df = add_eia_info(plant_name_ferc1="j h miller - total", plant_id_eia=6002, plant_name_eia='James H Miller Jr', generator_id='1')

In [19]:
# joseph m. farley
df = add_notes(plant_name_ferc1="joseph m. farley", notes='Two unit Nuclear plant')
df = add_eia_info(plant_name_ferc1="joseph m. farley", plant_id_eia=6001, plant_name_eia='Joseph M Farley', generator_id='1')

In [20]:
# lowndes county
p = "lowndes county"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7698, generator_id='1', units=2)
df = add_notes(plant_name_ferc1="lowndes county", notes="Don't see it EIA data. AL Power website says its a Cogen plant located at the SABIC plant in Burkville, west of Montgomery")

In [21]:
# mississippi power's, other power generation - corporate expenses, powersouth, steam power generation - corporate expenses
df = delete_plant_rows(plant_names=["mississippi power's","other power generation - corporate expenses","powersouth","steam power generation - corporate expenses"])

In [22]:
# theodore
df = add_notes(plant_name_ferc1="theodore", notes='2-unit NG combined cycle')
df = add_eia_info(plant_name_ferc1="theodore", plant_id_eia=7721, plant_name_eia='Theodore', generator_id='1')

In [23]:
# washington county
df = add_notes(plant_name_ferc1="washington county", notes='2-unit NG combined cycle')
df = add_eia_info(plant_name_ferc1="washington county", plant_id_eia=7697, plant_name_eia='Washington County Cogen Facility', generator_id='1')

### Allete, Inc.

In [24]:
# boswell
df = add_notes(plant_name_ferc1="boswell", notes='Two operational coal units, 364.5 MW and 558 MW respectively')
df = add_eia_info(plant_name_ferc1="boswell", plant_id_eia=1893, plant_name_eia='Clay Boswell', generator_id='3')

In [25]:
# hibbard
df = add_notes(plant_name_ferc1="hibbard", notes='2-unit biomass plant')
df = add_eia_info(plant_name_ferc1="hibbard", plant_id_eia=1897, plant_name_eia='M L Hibbard', generator_id='3')

In [26]:
# laskin
df = add_notes(plant_name_ferc1="laskin", notes='2-unit NG steam plant')
df = add_eia_info(plant_name_ferc1="laskin", plant_id_eia=1891, plant_name_eia='Syl Laskin', generator_id='1')

In [27]:
# rapids non rate-base
df = add_notes(plant_name_ferc1="rapids non rate-base", notes='Two NG steam units, one small hydro unit')
df = add_eia_info(plant_name_ferc1="rapids non rate-base", plant_id_eia=10686, plant_name_eia='Rapids Energy Center', generator_id='6')

In [28]:
p = "taconite harbor"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=10075, generator_id='GEN1', units=3)
df = is_retired(plant_name_ferc1=p)

### Alaska Electric Light and Power

In [29]:
p = "1 - lemon creek (ic)"
df = add_notes(plant_name_ferc1=p, notes='9 unit oil IC')
df = add_eia_info(plant_name_ferc1=p, plant_id_eia=64, plant_name_eia='Lemon Creek', generator_id='1')
p = "2 - lemon creek (gt)"
df = add_notes(plant_name_ferc1=p, notes='2-unit oil GT')
df = add_eia_info(plant_name_ferc1=p, plant_id_eia=64, plant_name_eia='Lemon Creek', generator_id='5')

In [30]:
p = "3 - auke bay (ic)"
df = add_eia_info(plant_name_ferc1=p, plant_id_eia=7250, plant_name_eia='Auke Bay', generator_id='4')
p = "4 - auke bay (gt)"
df = add_notes(plant_name_ferc1=p, notes='2-unit GT')
df = add_eia_info(plant_name_ferc1=p, plant_id_eia=7250, plant_name_eia='Auke Bay', generator_id='13')

In [31]:
p = "5 - industrial blvd (gt)"
df = add_eia_info(plant_name_ferc1=p, plant_id_eia=59793, plant_name_eia='Industrial Plant', generator_id='15')

### Alcoa Power Generating Inc.

In [32]:
p = "unit 4"
df = add_eia_info(plant_name_ferc1=p, plant_id_eia=6705, plant_name_eia='Warrick', generator_id='4')
p = "units 1-3"
df = add_eia_info(plant_name_ferc1=p, plant_id_eia=6705, plant_name_eia='Warrick', generator_id='1')

### Appalachian Power Company

In [33]:
p = "amos"
df = add_notes(plant_name_ferc1=p, notes="3-unit coal plant")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3935, generator_id='1')

In [34]:
p = "ceredo"
df = add_notes(plant_name_ferc1=p, notes="6-unit NG CT")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55276, generator_id='1')

In [35]:
p = "clinch river"
df = add_notes(plant_name_ferc1=p, notes="2-unit NG Steam plant")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3775, generator_id='1')

In [36]:
p = "dresden"
df = add_notes(plant_name_ferc1=p, notes="3-unit NGCC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55350, generator_id='1')

In [37]:
p = "mountaineer"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6264, generator_id='1')

### Arizona Public Service Company

In [38]:
p = "plant name: cholla 1 steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=113, generator_id='1')
p = "plant name: cholla 3 steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=113, generator_id='3')

In [39]:
p = "plant name: douglas comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=114, generator_id='1')

In [40]:
p = "plant name: four corners 4 steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2442, generator_id='4')
p = "plant name: four corners 5 steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2442, generator_id='5')

In [41]:
i = 116
p= "plant name: ocotillo 1 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT1')
p = "plant name: ocotillo 2 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT2')
p = "plant name: ocotillo 3 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT3')
p = "plant name: ocotillo 4 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT4')
p = "plant name: ocotillo 5 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT5')
p = "plant name: ocotillo 6 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT6')
p = "plant name: ocotillo 7 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT7')

In [42]:
p = "plant name: palo verde 1 nuclear"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6008, generator_id='1')
p = "plant name: palo verde 2 nuclear"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6008, generator_id='2')
p = "plant name: palo verde 3 nuclear"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6008, generator_id='3')

In [43]:
p = "plant name: redhawk 1 combined cycle"
df = add_notes(plant_name_ferc1=p, notes="3-unit NGCC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55455, generator_id='CT1A')
p = "plant name: redhawk 2 combined cycle"
df = add_notes(plant_name_ferc1=p, notes="3-unit NGCC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55455, generator_id='CT2A')

In [44]:
p = "plant name: saguaro 1 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=118, generator_id='GT1')
p = "plant name: saguaro 2 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=118, generator_id='GT2')
p = "plant name: saguaro 3 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=118, generator_id='GE1')

In [45]:
p = "plant name: sundance comb. turbine"
df = add_notes(plant_name_ferc1=p, notes="10-unit NGCT")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55522, generator_id='CT1')

In [46]:
i = 117
p = "plant name: west phoenix 1 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT1')
p = "plant name: west phoenix 2 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT2')
p = "plant name: west phoenix 1 combined cycle"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='1B')
p = "plant name: west phoenix 2 combined cycle"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='2B')
p = "plant name: west phoenix 3 combined cycle"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='3B')
p = "plant name: west phoenix 4 combined cycle"
df = add_notes(plant_name_ferc1=p, notes="2-unit NGCC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='C4-1')
p = "plant name: west phoenix 5 combined cycle"
df = add_notes(plant_name_ferc1=p, notes="3-unit NGCC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='C5-1')

In [47]:
i = 120
p = "plant name: yucca 1 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT1')
p = "plant name: yucca 2 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT2')
p = "plant name: yucca 3 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT3')
p = "plant name: yucca 4 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT4')
p = "plant name: yucca 5 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT5')
p = "plant name: yucca 6 comb. turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=i, generator_id='GT6')

### Avista Corporation

In [48]:
p = "boulder park"
df = add_notes(plant_name_ferc1=p, notes="6-unit NG combustion engine")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8022, generator_id='1')

In [49]:
p = "colstrip"
df = add_notes(plant_name_ferc1=p, notes="EIA lists plant under PP&L Montana LLC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6076, generator_id='3')

In [50]:
p = "coyote springs 2"
df = add_notes(plant_name_ferc1=p, notes="EIA lists it as a 2-unit NGCC. Coyote Springs 1 is owned by PGE, listed as separate plant in same location")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7931, generator_id='1')

In [51]:
p = "kettle falls"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=550, generator_id='1')

In [52]:
p = "rathdrum"
df = add_notes(plant_name_ferc1=p, notes="2-unit NGCT")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6210, generator_id='1', units=1)

In [53]:
p = "spokane n.e."
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7456, generator_id='1')
df = add_notes(plant_name_ferc1=p, notes="Can't find it in EIA table. The city of Spokane is listed with a smaller municipal solid waste plant of same name")

### Black Hills Power, Inc.

In [54]:
p = "ben french diesel"
df = add_notes(plant_name_ferc1=p, notes="5-unit diesel generator")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3325, generator_id='1')
p = "ben french station"
df = add_notes(plant_name_ferc1=p, notes="4-unit NGCT")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3325, generator_id='GT1')

In [55]:
p = "cheyenne prairie generating station"
df = add_notes(plant_name_ferc1=p, notes="EIA lists a 3-unit NGCC and 1-unit NGCT (with two more NGCT units proposed). Existing plant has larger capacity in EIA (140MW) than FERC data")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=57703, generator_id='01A')

In [56]:
p = "lange ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55478, generator_id='1')

In [57]:
p = "neil simpson ct #1"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7504, generator_id='GT1')
p  = "neil simpson unit 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7504, generator_id='2')

In [58]:
p = "wygen 3"
df = add_notes(plant_name_ferc1=p, notes="3 different coal plants with name 'Wygen', all in same county, each has has larger capacity than listed in FERC table")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56596, generator_id='5')

In [59]:
p = "wyodak"
df = add_notes(plant_name_ferc1=p, notes="EIA lists plant owned by PacifiCorp")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6101, generator_id='1')

### Black Hills/Colorado Electric Utility Company, LP

In [60]:
p = "airport diesel"
df = add_notes(plant_name_ferc1=p, notes="4 generating units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7995, generator_id='IC1')

In [61]:
p = "pueblo airport generating station- unit 6"
df = add_notes(plant_name_ferc1=p, notes="No CT's listed under this kind of name. There is a retired NG steam and the diesel ICs, just going to assign it to their location")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=460, generator_id='6')
df = is_retired(plant_name_ferc1=p)

p = "pueblo airport generation station- unit 1 & 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=460, generator_id='IC3')
p = "pueblo diesels"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=460, generator_id='IC4')

### CENTRAL HUDSON GAS & ELECTRIC CORPORATION

In [62]:
p = "coxsackie"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2487, generator_id='GT1')

In [63]:
p = "south cairo"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2485, generator_id='GT1')

### Cheyenne Light, Fuel and Power Company

In [64]:
p = "clfp 42% share pb1 cheyenne prairie generating station"
df = add_notes(plant_name_ferc1=p, notes="Shares the plant with Black Hills")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=57703, generator_id='01B')
p = "unit 02a cheyenne prairie generating station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=57703, generator_id='02A')

In [65]:
p = "wygen 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56319, generator_id='1')

### Chugach Electric Association, Inc.

In [66]:
p = "beluga"
df = add_notes(plant_name_ferc1=p, notes="7-unit NGCT")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=96, generator_id='1')

In [67]:
p = "igt"
df = add_notes(plant_name_ferc1=p, notes="Likely a 3-unit NGCT, listed retired in EIA 2023")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6293, generator_id='1')
df = is_retired(plant_name_ferc1=p)

In [68]:
p = "nikkels"
df = add_notes(plant_name_ferc1=p, notes="2-unit NGCT")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=75, generator_id='3R')

In [69]:
p = "spp"
df = add_notes(plant_name_ferc1=p, notes="EIA lists as 4-unit NGCC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=57036, generator_id='1')

In [70]:
p = "sullivan"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6559, generator_id='7', units=7)
df = add_notes(plant_name_ferc1=p, notes="EIA lists a 6-unit NG plant")

### Cleco Power LLC

In [71]:
p = "acadia unit 1"
df = add_notes(plant_name_ferc1=p, notes="3-unit NGCC. There is a second 3-unit NGCC plant of identical size that EIA lists with CLECO, but is not included under Cleco in FERC1")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55173, generator_id='CT11')

In [72]:
p = "coughlin"
df = add_notes(plant_name_ferc1=p, notes="5-unit NGCC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1396, generator_id='6')

In [73]:
p = "dolet hills power station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=51, generator_id='1')

In [74]:
p = "madison unit 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6190, generator_id='3')

In [75]:
p = "nesbitt unit 1"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6190, generator_id='1')

In [76]:
p = "rodemacher unit 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6190, generator_id='2')

In [77]:
p = "st. mary's"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60610, generator_id='1')

In [78]:
p = "teche unit 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1400, generator_id='3')
p = "teche unit 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1400, generator_id='4')

### Consolidated Edison Company of New York, Inc.

In [79]:
p = "59th st gt-1"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2503, generator_id='GT1')

In [80]:
p = "74th st gt 1 & 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2504, generator_id='GT1')

In [81]:
p = "east river 6&7"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2493, generator_id='6')

In [82]:
p = "hudson ave gt 3,4 & 5"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2496, generator_id='GT3')

### Consumers Energy Company

In [83]:
p = "campbell 1 &2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1710, generator_id='1')

df = delete_plant_rows(plant_names=["campbell 3 (ceco)"])

p = "campbell 3 (total)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1710, generator_id='3')

In [84]:
p = "covert"
df = add_notes(plant_name_ferc1=p, notes="6 unit NGCC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55297, generator_id='1')

In [85]:
p = "jackson gas plant"
df = add_notes(plant_name_ferc1=p, notes="9 unit NGCC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55270, generator_id='7EA')

In [86]:
p = "karn 1 & 2"
df = add_notes(plant_name_ferc1=p, notes="EIA 2023 table says they're retired")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1702, generator_id='1A')
df = is_retired(plant_name_ferc1=p)

In [87]:
p = "karn 3 & 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1702, generator_id='3')

In [88]:
p = "zeeland"
df = add_notes(plant_name_ferc1=p, notes="5-unit NGCC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55087, generator_id='1A')

### DTE Electric Company

In [89]:
p = "belle river (total)"
df = add_notes(plant_name_ferc1=p, notes="2-unit coal plant")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6034, generator_id='ST1')
df = delete_plant_rows(plant_names=["belle river dte-81%"])

p = "belle river gas pkr"
df = add_notes(plant_name_ferc1=p, notes="3-unit NGCT")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6034, generator_id='13-1')

p = "belle river oil pkr"
df = add_notes(plant_name_ferc1=p, notes="5-units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6034, generator_id='IC1')

In [90]:
p = "blue water energy center"
df = add_notes(plant_name_ferc1=p, notes="3-unit NGCC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62192, generator_id='11')

In [91]:
p = "colfax peaker"
df = add_notes(plant_name_ferc1=p, notes="5-unit IC plant")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1725, generator_id='1')

In [92]:
p = "dean peaker"
df = add_notes(plant_name_ferc1=p, notes="4-units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55718, generator_id='GT1')

In [93]:
p = "dearborn energy center"
df = add_notes(plant_name_ferc1=p, notes="3-unit NGCC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62289, generator_id='GTG-1')

In [94]:
p = "delray peaker"
df = add_notes(plant_name_ferc1=p, notes="2-units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1728, generator_id='11-1')

In [95]:
p = "enrico fermi peaker"
df = add_notes(plant_name_ferc1=p, notes="4-units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1729, generator_id='GT1')

In [96]:
p = "fermi 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1729, generator_id='2')

In [97]:
p = "greenwood ec"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6035, generator_id='1')

In [98]:
p = "greenwood peaker"
df = add_notes(plant_name_ferc1=p, notes="3-units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6035, generator_id='11-1')

In [99]:
p = "hancock peaker"
df = add_notes(plant_name_ferc1=p, notes="6 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1730, generator_id='1')

In [100]:
p = "monroe"
df = add_notes(plant_name_ferc1=p, notes="4-units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1733, generator_id='1')

In [101]:
p = "monroe peaker"
df = add_notes(plant_name_ferc1=p, notes="5 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1733, generator_id='IC1')

In [102]:
p = "northeast peaker"
df = add_notes(plant_name_ferc1=p, notes="5-units (87.4MW) NGCT, 2 units (42.4MW) petroleum liquids ")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1734, generator_id='1')

In [103]:
p = "oliver peaker"
df = add_notes(plant_name_ferc1=p, notes="5 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1735, generator_id='1')

In [104]:
p = "placid peaker"
df = add_notes(plant_name_ferc1=p, notes="5-units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1737, generator_id='1')

In [105]:
p = "putnam peaker"
df = add_notes(plant_name_ferc1=p, notes="5 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1739, generator_id='1')

In [106]:
p = "renaissance peaker"
df = add_notes(plant_name_ferc1=p, notes="4 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55402, generator_id='CT1')

In [107]:
p = "river rouge peaker"
df = add_notes(plant_name_ferc1=p, notes="4-units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1740, generator_id='IC1')

In [108]:
p = "slocum peaker"
df = add_notes(plant_name_ferc1=p, notes="5-units, EIA table says they're retired")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1741, generator_id='1', units=5)
df = is_retired(plant_name_ferc1=p)

In [109]:
p = "st. clair peaker"
df = add_notes(plant_name_ferc1=p, notes="Plant has 1 CT unit (18.5 MW) plus two small petroleum liquids units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1743, generator_id='11')

In [110]:
p = "superior peaker"
df = add_notes(plant_name_ferc1=p, notes="4-units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1744, generator_id='1')

In [111]:
p = "wilmot peaker"
df = add_notes(plant_name_ferc1=p, notes="5 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1746, generator_id='1')

### Deseret Generation & Transmission Cooperative

In [112]:
p = "bonanza (dgt share)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7790, generator_id='1')

In [113]:
p = "hunter ii (dgt share)"
df = add_notes(plant_name_ferc1=p, notes="Deseret owns less than 1 unit of capacity. PacifiCorp owns majority of this 3-unit plant and reports its share of costs separately")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6165, generator_id='1')

In [114]:
p = "solomon facility"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7790, generator_id='1')

### Duke Energy Carolinas, LLC

In [115]:
p = "allen"
df = add_notes(plant_name_ferc1=p, notes="2 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2718, generator_id='1')

In [116]:
p = "belews creek"
df = add_notes(plant_name_ferc1=p, notes="2-units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8042, generator_id='1')

In [117]:
p = "buck" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2720, generator_id='3', units=10)
df = is_retired(plant_name_ferc1=p)

p = "buck cc"
df = add_notes(plant_name_ferc1=p, notes="3 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2720, generator_id='CT11')

p = "buck ct" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2720, generator_id='7', units=10)
df = is_retired(plant_name_ferc1=p)

In [118]:
p = "buzzard roost" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=58212, generator_id='6', units=10)
df = is_retired(plant_name_ferc1=p)

In [119]:
p = "catawba"
df = add_notes(plant_name_ferc1=p, notes="2 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6036, generator_id='1')

In [120]:
p = "clemson chp"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=63063, generator_id='GT01')

In [121]:
p = "cliffside"
df = add_notes(plant_name_ferc1=p, notes="2 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2721, generator_id='5')

In [122]:
p = "dan river" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2723, generator_id='4', units=9)
df = is_retired(plant_name_ferc1=p)

p = "dan river cc"
df = add_notes(plant_name_ferc1=p, notes="3 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2723, generator_id='CT8')

p = "dan river steam" # Retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2723, generator_id='1', units=9)
df = is_retired(plant_name_ferc1=p)

In [123]:
p = "lee"
df = add_notes(plant_name_ferc1=p, notes="2 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3264, generator_id='7')

p = "lee cc"
df = add_notes(plant_name_ferc1=p, notes="3-units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3264, generator_id='CT11')

p = "lee steam" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2709, generator_id='1', units=7)
df = is_retired(plant_name_ferc1=p)

In [124]:
p = "lincoln"
df = add_notes(plant_name_ferc1=p, notes="16 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7277, generator_id='1')

In [125]:
p = "marshall"
df = add_notes(plant_name_ferc1=p, notes="4 unit steam plant. 2 units (half the capacity) are NG fueled and other 2 units are coal")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2727, generator_id='1')

In [126]:
p = "mcguire"
df = add_notes(plant_name_ferc1=p, notes="2 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6038, generator_id='1')

In [127]:
p = "millcreek"
df = add_notes(plant_name_ferc1=p, notes="8 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7981, generator_id='1')

In [128]:
p = "oconee"
df = add_notes(plant_name_ferc1=p, notes="3 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3265, generator_id='1')

In [129]:
p = "riverbend" # Retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2732, generator_id='8', units=8)
df = is_retired(plant_name_ferc1=p)

p = "riverbend steam" # Retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2732, generator_id='4', units=8)
df = is_retired(plant_name_ferc1=p)

In [130]:
p = "rockingham"
df = add_notes(plant_name_ferc1=p, notes="5 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55116, generator_id='CT1')

### Duke Energy Florida, Inc.

In [131]:
p = "anclote"
df = add_notes(plant_name_ferc1=p, notes="2 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8048, generator_id='1')

In [132]:
p = "bartow"
df = add_notes(plant_name_ferc1=p, notes="5 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=634, generator_id='4AGT')

In [133]:
p = "bartow ct"
df = add_notes(plant_name_ferc1=p, notes="4 units, half gas half fuel oil")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=634, generator_id='P1')

In [134]:
p = "bayboro"
df = add_notes(plant_name_ferc1=p, notes="4 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=627, generator_id='P1')

In [135]:
p = "citrus county"
df = add_notes(plant_name_ferc1=p, notes="6 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=628, generator_id='1GTA')

In [136]:
p = "crystal river north"
df = add_notes(plant_name_ferc1=p, notes="2 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=628, generator_id='5')

In [137]:
p = "debary"
df = add_notes(plant_name_ferc1=p, notes="9 units, some are NG and some fuel oil")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6046, generator_id='2')

In [138]:
p = "hines"
df = add_notes(plant_name_ferc1=p, notes="12 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7302, generator_id='1GT')

In [139]:
p = "intercession city"
df = add_notes(plant_name_ferc1=p, notes="14 units, some gas and some fuel oil")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8049, generator_id='P7')

In [140]:
p = "osprey"
df = add_notes(plant_name_ferc1=p, notes="3 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55412, generator_id='OEC1')

In [141]:
p = "suwannee"
df = add_notes(plant_name_ferc1=p, notes="3 units, 2 are NGCT, one diesel fuel")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=638, generator_id='P1')

In [142]:
p = "tiger bay"
df = add_notes(plant_name_ferc1=p, notes="2 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7699, generator_id='CT1')

In [143]:
p = "university of florida"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7345, generator_id='P1')

### Duke Energy Indiana, Inc.

In [144]:
p = "cadiz (henry county)"
df = add_notes(plant_name_ferc1=p, notes="3 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7763, generator_id='1')

In [145]:
p = "cayuga"
df = add_notes(plant_name_ferc1=p, notes="2 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1001, generator_id='1')

p = "cayuga ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1001, generator_id='4')

p = "cayuga peaking"
df = add_notes(plant_name_ferc1=p, notes="4 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1001, generator_id='31')

In [146]:
p = "connersville" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1002, generator_id='1', units=2)
df = is_retired(plant_name_ferc1=p)

In [147]:
p = "edwardsport" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1004, generator_id='7', units=6)
df = is_retired(plant_name_ferc1=p)

p = "edwardsport igcc"
df = add_notes(plant_name_ferc1=p, notes="3 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1004, generator_id='CT1')

In [148]:
p = "gallagher" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1008, generator_id='1', units=4)
df = is_retired(plant_name_ferc1=p)

In [149]:
p = "gibson"
df = add_notes(plant_name_ferc1=p, notes="5 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6113, generator_id='1')

In [150]:
p = "madison"
df = add_notes(plant_name_ferc1=p, notes="8 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55110, generator_id='CT1')

In [151]:
p = "miami wabash" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1006, generator_id='1', units=6)
df = is_retired(plant_name_ferc1=p)

In [152]:
p = "noblesville"
df = add_notes(plant_name_ferc1=p, notes="5 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1007, generator_id='1')

In [153]:
p = "purdue chp"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65400, generator_id='PUCHP')

In [154]:
p = "vermillion"
df = add_notes(plant_name_ferc1=p, notes="6 units, plant ownership is likely shared")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55111, generator_id='CT1')

In [155]:
p = "wabash river" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1010, generator_id='2', units=8)
df = is_retired(plant_name_ferc1=p)



In [156]:
p = "wheatland"
df = add_notes(plant_name_ferc1=p, notes="4 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55224, generator_id='CTG1')

### Duke Energy Kentucky, Inc.

In [157]:
p = "east bend"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6018, generator_id='2')

In [158]:
p = "miami fort 6" # Retired, other coal and fuel oil units are at this plant associated with Vistra Energy
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2832, generator_id='6', units=8)
df = is_retired(plant_name_ferc1=p)

In [159]:
p = "woodsdale ct"
df = add_notes(plant_name_ferc1=p, notes="6 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7158, generator_id='GT1')

### Duke Energy Progress, Inc.

In [160]:
p = "asheville cc"
df = add_notes(plant_name_ferc1=p, notes="4 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2706, generator_id='CT5')

p = "asheville gas turbine"
df = add_notes(plant_name_ferc1=p, notes="2 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2706, generator_id='GT1')

p = "asheville steam" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2706, generator_id='1', units=8)
df = is_retired(plant_name_ferc1=p)

In [161]:
p = "blewett"
df = add_notes(plant_name_ferc1=p, notes="4 units")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2707, generator_id='GT1')

In [162]:
p = "brunswick"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6014, generator_id='1', units=2)

p = "cape fear gas turbine" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2708, generator_id='1', units=8)
df = is_retired(plant_name_ferc1=p)

p = "cape fear steam" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2708, generator_id='5', units=8)
df = is_retired(plant_name_ferc1=p)

p = "darlington"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3250, generator_id='12', units=2)

p = "h.b. robinson gas turbine" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3251, generator_id='GT1', units=3)
df = is_retired(plant_name_ferc1=p)

p = "h.b. robinson nuclear"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3251, generator_id='2', units=1)

p = "h.b. robinson steam" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3251, generator_id='1', units=3)
df = is_retired(plant_name_ferc1=p)

p = "h.f. lee gas turbine"
df = add_notes(plant_name_ferc1=p, notes="Listed as CT plant in FERC, CC in EIA")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=58215, generator_id='1A', units=4)

p = "h.f. lee steam" #retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2709, generator_id='1', units=7)
df = is_retired(plant_name_ferc1=p)

p = "harris"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6015, generator_id='1', units=1)

p = "l.v. sutton gas turbine"
df = add_notes(plant_name_ferc1=p, notes="CT in FERC, majority is CC in EIA, but also includes 2 CT units in EIA")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=58697, generator_id='CA1', units=5)

p = "l.v. sutton steam" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2713, generator_id='1', units=6)
df = is_retired(plant_name_ferc1=p)

p = "mayo"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6250, generator_id='1', units=1)

p = "morehead" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2711, generator_id='GT1', units=2)
df = is_retired(plant_name_ferc1=p)

p = "roxboro"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2712, generator_id='1', units=5)

p = "smith energy complex"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7805, generator_id='1', units=11)

p = "w.h. weatherspoon gas turbine"
df = add_notes(plant_name_ferc1=p, notes="FERC capacity is 40 MW, EIA shows 4 fuel oil units at 40 MW a piece")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2716, generator_id='GT1', units=4)

p = "w.h. weatherspoon steam" # Retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2716, generator_id='1', units=7)
df = is_retired(plant_name_ferc1=p)

p = "wayne county"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7538, generator_id='1', units=5)

### El Paso Electric Company

In [163]:
p = "copper"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=9, generator_id='1', units=1)

p = "montana"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=58562, generator_id='GT-1', units=4)

p = "newman"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3456, generator_id='1', units=6)

p = "newman unit 6"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3456, generator_id='6', units=1)

p = "palo verde"
df = add_notes(plant_name_ferc1=p, notes="Plant shared with several other partners, associated with APS in EIA")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6008, generator_id='1', units=3)

p = "rio grande"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2444, generator_id='6', units=3)

p = "rio grande unit 9"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2444, generator_id='9', units=1)

### Entergy Arkansas, Inc.

In [164]:
p = "arkansas nuclear one"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8055, generator_id='1', units=2)

p = "hot spring"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55418, generator_id='CT1', units=3)

p = "independence"
df = add_notes(plant_name_ferc1=p, notes="FERC lists plant capacity at 284, although EIA reports its an 1800 coal plant")
df = data_questionable(plant_name_ferc1=p)
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6641, generator_id='1', units=2)

p = "lake catherine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=170, generator_id='4', units=1)

p = "ouachita 1 & 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55467, generator_id='CTG1', units=4)
df = add_notes(plant_name_ferc1=p, notes="FERC submission split between Entergy AK and Entergy LA")

p = "union power station"
df = add_notes(plant_name_ferc1=p, notes="FERC data splits plant between Entergy AK and Entergy NOLA")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55380, generator_id='CTG1', units=12)

p = "white bluff"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6009, generator_id='1', units=2)

### Entergy Louisiana, LLC

In [165]:
p = "acadia"
df = add_notes(plant_name_ferc1=p, notes="Cleco submits costs to FERC for the other portion of this plant")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55173, generator_id='CT11', units=6)

p = "big cajun 2 unit 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6055, generator_id='3', units=1)

p = "calcasieu"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55165, generator_id='G101', units=2)

p = "j. wayne leonard"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60926, generator_id='1A', units=3)
df = add_notes(plant_name_ferc1=p, notes="EIA reports total plant capacity of 1000, but 1280MW is reported in FERC")

p = "lake charles"
df = add_notes(plant_name_ferc1=p, notes="EIA reports total plant capacity of 1000, but 1280MW is reported in FERC")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60927, generator_id='1A', units=3)

p = "little gypsy 2 & 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1402, generator_id='1', units=2)

p = "louisiana station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1392, generator_id='10', units=3)
df = add_notes(plant_name_ferc1=p, notes="EIA reports 175MW capacity, FERC reports 250MW")

p = "ninemile 6"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1403, generator_id='6A', units=3)

p = "ninemile point 4& 5"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1403, generator_id='5', units=2)

p = "ouachita 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55467, generator_id='CTG3', units=2)

p = "perryville"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55620, generator_id='2-CT', units=4)

p = "river bend"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6462, generator_id='1', units=1)

p = "roy s. nelson 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1393, generator_id='4', units=1)

p = "roy s. nelson 6"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1393, generator_id='6', units=1)

p = "sterlington 7"
df = add_notes(plant_name_ferc1=p, notes="EIA shows most of these plants are retired, except for 60 MW GT online. FERC lists 160MW CC")
df = data_questionable(plant_name_ferc1=p)
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1404, generator_id='7A', units=3)

p = "union 3 & 4"
df = add_notes(plant_name_ferc1=p, notes="Shared with Entergy Arkansas")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55380, generator_id='CTG3', units=12)

p = "washington parish"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55486, generator_id='CTG01', units=2)

p = "waterford 1 & 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8056, generator_id='1', units=3)
df = add_notes(plant_name_ferc1=p, notes="3rd unit is a smaller fuel oil GT")

p = "waterford 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4270, generator_id='3', units=1)

### Entergy Texas, Inc.

In [166]:
p = "big cajun 2 unit 3"
df = add_notes(plant_name_ferc1=p, notes="Small portion of the plant")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6055, generator_id='3', units=1)

p = "hardin county"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56604, generator_id='HC1', units=2)

p = "lewis creek"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3457, generator_id='1', units=2)

p = "montgomery county"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60925, generator_id='1A', units=3)

p = "roy s. nelson 6"
df = add_notes(plant_name_ferc1=p, notes="Portion of the plant shared w  Entergy LA")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1393, generator_id='6', units=1)

p = "sabine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3459, generator_id='1', units=5)

### Florida Power & Light Company

In [167]:
p = "anhinga"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65431, generator_id='1', units=1)

p = "apalachee"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65432, generator_id='1', units=1)

p = "babcock preserve solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62634, generator_id='1', units=1)

p = "babcock solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59993, generator_id='1', units=2)

p = "barefoot bay solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61051, generator_id='1', units=1)

p = "blackwater river"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65433, generator_id='1', units=1)

p = "blue cypress solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61029, generator_id='1', units=1)

p = "blue heron solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62631, generator_id='1', units=1)

p = "blue indigo solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=63754, generator_id='SBI01', units=1)

p = "blue springs solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64757, generator_id='1', units=1)

p = "bluefield preserve"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65420, generator_id='1', units=1)

p = "cape canaveral"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=609, generator_id='3A', units=4)

p = "cattle ranch solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62632, generator_id='1', units=1)

p = "cavendish"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65438, generator_id='1', units=1)

p = "chautauqua"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65421, generator_id='1', units=1)

p = "chipola river"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65422, generator_id='1', units=1)

p = "citrus solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60061, generator_id='1', units=2)

p = "coral farms solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61022, generator_id='1', units=1)

p = "cotton creek solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65036, generator_id='1', units=1)

p = "cypress pond"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65593, generator_id='1', units=1)

p = "dania beach energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65978, generator_id='7GT1', units=3)

p = "daniel"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6073, generator_id='1', units=8)
df = add_notes(plant_name_ferc1=p, notes="FPL has one of the coal units, rest of the plant (including a combined cycle plant) is assigned to Mississippi Power Co")

p = "desoto solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56929, generator_id='1', units=1)

p = "discovery solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=63109, generator_id='1', units=1)

p = "echo river solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62490, generator_id='1', units=2)

p = "egret solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62925, generator_id='1', units=1)

p = "elder branch solar engery center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65041, generator_id='1', units=1)

p = "etonia creek"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65594, generator_id='1', units=1)

p = "evans grove solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65042, generator_id='1', units=1)

p = "everglades"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65423, generator_id='1', units=1)

p = "first city"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65424, generator_id='1', units=1)

p = "flowers creek"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65427, generator_id='1', units=1)

p = "fort drum solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64300, generator_id='FORTD', units=1)

p = "ft myers"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=612, generator_id='2A', units=8)

p = "ft myers - gas"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=612, generator_id='GT1', units=2)

p = "ft myers - simple cycle"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=612, generator_id='CT1', units=4)

p = "ghost orchid solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65038, generator_id='1', units=1)

p = "gulf clean energy center 4-7"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=641, generator_id='4', units=4)

p = "gulf clean energy center 8"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=641, generator_id='8A', units=4)

p = "hammock solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61024, generator_id='1', units=1)

p = "hibiscus solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62206, generator_id='1', units=1)

p = "horizon solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61021, generator_id='1', units=1)

p = "immokalee solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65040, generator_id='1', units=1)

p = "indian river solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61020, generator_id='1', units=1)

p = "interstate solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61768, generator_id='1', units=1)

p = "lakeside solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62922, generator_id='1', units=1)

p = "lauderdale"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=613, generator_id='3', units=2)

p = "lauderdale - simple cycle"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=613, generator_id='6A', units=5)

p = "loggerhead solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61052, generator_id='1', units=1)

p = "magnolia springs solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62915, generator_id='1', units=1)

p = "manatee - cc" 
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6042, generator_id='A', units=5)

p = "manatee - steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6042, generator_id='1', units=2)

p = "manatee solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60014, generator_id='1', units=2)

p = "martin 3&4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6043, generator_id='3GT1', units=6)

p = "martin 8"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6043, generator_id='8', units=5)

p = "martin solar energy center" # A solar thermal facility co located at martin NGCC plant
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6043, generator_id='8', units=5)

p = "miami-dade solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61766, generator_id='1', units=1)

p = "nassau solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62914, generator_id='1', units=1)

p = "northern preserve solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62645, generator_id='1', units=1)

p = "okeechobee"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60345, generator_id='1A', units=4)

p = "okeechobee solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62491, generator_id='1', units=1)

p = "orange blossom solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62919, generator_id='1', units=1)

p = "palm bay solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62921, generator_id='1', units=1)

p = "pea ridge"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7715, generator_id='1', units=3)

p = "pelican solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62924, generator_id='1', units=1)

p = "pink trail"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65428, generator_id='1', units=1)

p = "pioneer trail solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61767, generator_id='1', units=1)

p = "port everglades"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=617, generator_id='5A', units=4)

p = "riviera"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=619, generator_id='5A', units=4)

p = "rodeo solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62917, generator_id='1', units=1)

p = "sabal palm solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=63110, generator_id='1', units=1)

p = "sanford"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=620, generator_id='4', units=10)

p = "saw palmetto"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65592, generator_id='1', units=1)

p = "sawgrass solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65037, generator_id='1', units=1)

p = "scherer 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6257, generator_id='3', units=3)
df = add_notes(plant_name_ferc1=p, notes="Associated with Georgia Power Co")

p = "shirer branch"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65429, generator_id='1', units=1)

p = "smith 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=643, generator_id='3A', units=3)

p = "smith a"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=643, generator_id='CT1', units=1)

p = "southfork solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62493, generator_id='1', units=1)

p = "space coast solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56930, generator_id='1', units=1)

p = "st. lucie - nuclear"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6045, generator_id='1', units=2)

p = "sundew solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65039, generator_id='1', units=1)

p = "sunshine gateway solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61763, generator_id='1', units=2)

p = "sweetbay solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62394, generator_id='1', units=1)

p = "trailside solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62916, generator_id='1', units=1)

p = "turkey point"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=621, generator_id='5CA', units=5)

p = "turkey point - nuclear"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=621, generator_id='3', units=2)

p = "twin lakes solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62633, generator_id='1', units=1)

p = "union springs solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62923, generator_id='1', units=1)

p = "west county energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56407, generator_id='1A', units=12)

p = "wild azalea"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65430, generator_id='1', units=1)

p = "wildflower solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61050, generator_id='1', units=1)

p = "willow solar energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64301, generator_id='WILLO', units=1)

### Georgia Power Company

In [168]:
p = "bowen"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=703, generator_id='1', units=4)

p = "fort benning"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59862, generator_id='1', units=1)

p = "fort gordon"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59863, generator_id='1', units=1)

p = "fort stewart"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59865, generator_id='1', units=1)

p = "fort valley state university"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=63062, generator_id='1', units=1)

p = "hatch"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6051, generator_id='1', units=1)
df = add_notes(plant_name_ferc1=p, notes="FERC only reports capacity for one half of the two units reported in EIA")

p = "king's bay"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59864, generator_id='1', units=1)

p = "mcdonough no. 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=710, generator_id='3A', units=2)

p = "mcdonough no. 4-6"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=710, generator_id='4', units=9)

p = "mcintosh (cc)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56150, generator_id='10ST', units=6)

p = "mcintosh (ct)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6124, generator_id='CT1', units=8)

p = "mclb"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59876, generator_id='1', units=1)

p = "mcmanus 3 & 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=715, generator_id='3A', units=9)

p = "moody afb"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62377, generator_id='1', units=1)

p = "robins afb"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7348, generator_id='1', units=1)

p = "scherer"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6257, generator_id='1', units=3)

p = "vogtle 1&2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=649, generator_id='1', units=2)

p ="vogtle 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=649, generator_id='3', units=2)

p = "warner robins"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7348, generator_id='1', units=2)

p = "wilson"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6258, generator_id='5A', units=7)

p = "yates"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=728, generator_id='6', units=2)

### Golden Spread Electric Cooperative, Inc.

In [169]:
p = "antelope station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=57865, generator_id='E01', units=18)

p = "elk station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=58835, generator_id='ELK1', units=3)

p = "mustang station cc"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55065, generator_id='GEN1', units=3)

p = "mustang station gt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56326, generator_id='GEN1', units=3)

### Green Mountain Power Corp

In [170]:
p = "ascutney gt #200"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3708, generator_id='GT4', units=1)

p = "berlin #005"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3734, generator_id='GT1', units=1)

p = "colchester #016"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3735, generator_id='GT1', units=1)

p = "mcneil #024"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=589, generator_id='1', units=1)

p = "rutland gt #201"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3723, generator_id='GT5', units=1)

p = "stony brook #096"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6081, generator_id='1', units=10)

p = "wyman #095"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1507, generator_id='1', units=5)

### Idaho Power Company

In [171]:
p = "bennett mountain"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7953, generator_id='1', units=3)

p = "boardman" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6106, generator_id='1', units=1)
df = is_retired(plant_name_ferc1=p)

p = "danskin"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55733, generator_id='1', units=1)

p = "jim bridger"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8066, generator_id='1', units=4)

p = "langley gulch"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=57028, generator_id='GTG', units=2)

p = "valmy"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8224, generator_id='1', units=2)

### Indiana Michigan Power Company

In [172]:
p = "donald c cook plant"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6000, generator_id='1', units=2)
# Rockport cleaned for IN MI under the AEP entry

### Indianapolis Power & Light Company

In [173]:
p = "eagle valley ccgt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=991, generator_id='GT1', units=3)

p = "georgetown"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7759, generator_id='GT1', units=4)

p = "harding street"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=990, generator_id='5', units=3)

p = "harding street gas turb"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=990, generator_id='GT1', units=5)

p = "petersburg"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=994, generator_id='4', units=5)

### Interstate Power and Light Company

In [174]:
p = "burlington"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1104, generator_id='1', units=1)

p = "burlington ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1104, generator_id='GT1', units=4)

p = "emery"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8031, generator_id='11', units=3)

p = "english farms wind"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61565, generator_id='1', units=1)

p = "franklin county"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=57844, generator_id='1', units=1)

p = "george neal #3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1091, generator_id='3', units=1)

p = "george neal #4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7343, generator_id='4', units=1)

p = "golden plains wind"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62081, generator_id='1', units=1)

p = "lansing #4" # Retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1047, generator_id='4', units=6)
df = is_retired(plant_name_ferc1=p)

p = "lime creek"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7155, generator_id='1', units=2)

p = "louisa"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6664, generator_id='1', units=1)

p = "marshalltown - mgs"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=58236, generator_id='CTG1', units=3)

p = "marshalltown ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1068, generator_id='3', units=3)

p = "ottumwa"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6254, generator_id='1', units=1)

p = "prairie creek 1, 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1073, generator_id='1', units=2)

p = "prairie creek 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1073, generator_id='4', units=1)

p = "richland wind"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62080, generator_id='1', units=1)

p = "upland prairie wind"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61564, generator_id='1', units=1)

p = "whispering willow east"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56355, generator_id='1', units=1)

p = "whispering willow north"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62079, generator_id='1', units=1)

### KCP&L Greater Missouri Operations Company

In [175]:
p = "crossroads"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55395, generator_id='CT01', units=4)

p = "greenwood"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6074, generator_id='1', units=5)

p = "iatan 1 (18%)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6065, generator_id='1', units=2)

p = "iatan 2 (18%)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6065, generator_id='2', units=2)

p = "jeffrey energy ctr 8%"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6068, generator_id='1', units=3)

p = "lake road - gas turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2098, generator_id='5', units=7)

p = "lake road - steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2098, generator_id='1', units=7)

p = "nevada"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2090, generator_id='1', units=1)

p = "ralph green"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2092, generator_id='3', units=1)

p = "sibley" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2094, generator_id='1', units=3)
df = is_retired(plant_name_ferc1=p)

p = "south harper"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56151, generator_id='GT1', units=3)

### Kansas City Power & Light Company

In [176]:
p = "hawthorn 5"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2079, generator_id='5', units=5)

p = "hawthorn 6 & 9"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2079, generator_id='6', units=5)

p = "hawthorn 7 & 8"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2079, generator_id='7', units=5)

p = "iatan 1 (100%)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6065, generator_id='1', units=2)

p = "iatan 2 (100%)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6065, generator_id='2', units=2)

df = delete_plant_rows(plant_names=["iatan 1 (18%)]", "iatan 2 (18%)","iatan 1 (70%)","iatan 2 (54.71%)", "iatan (1&2)"]) # Redundant with the plant entries above

p = "lacygne (100%)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1241, generator_id='1', units=2)

df = delete_plant_rows(plant_names=["lacygne 1 (50%)","lacygne 2 (50%)"]) # Redundant with entry above

p = "montrose" # retired
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2080, generator_id='1', units=3)
df = is_retired(plant_name_ferc1=p)

p = "northeast"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2081, generator_id='11', units=9)

p = "osawatomie"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7928, generator_id='1', units=1)

p = "west gardner"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7929, generator_id='1', units=4)

p = "wolf creek (47%)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=210, generator_id='1', units=1)


### Kentucky Power Company

In [177]:
p = "big sandy"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1353, generator_id='1', units=1)

p = "mitchell- total"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3948, generator_id='1', units=2)

df = delete_plant_rows(plant_names=["mitchell-kepco share","mitchell -total","mitchell-wpco share"])

p = "brown ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1355, generator_id='5', units=9)

p = "cane run ngcc"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1363, generator_id='7A', units=3)

p = "ew brown"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1355, generator_id='3', units=9)

p = "ghent"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1356, generator_id='1', units=4)

p = "haefling"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1358, generator_id='1', units=2)

p = "paddy's run 13 ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1366, generator_id='12', units=2)

p = "trimble county"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6071, generator_id='1', units=8)

p = "trimble county ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6071, generator_id='5', units=8)

### Louisville Gas and Electric Company

In [178]:
p = "mill creek"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1364, generator_id='1', units=4)

p = "paddy's run ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1366, generator_id='12', units=2)

### MDU Resources Group, Inc.

In [179]:
p = "big stone"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6098, generator_id='1', units=2)

p = "coyote"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8222, generator_id='1', units=1)

p = "glendive"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2176, generator_id='GT1', units=2)

p = "heskett iii"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2790, generator_id='3', units=1)

p = "lewis & clark ii"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6089, generator_id='2', units=2)

p = "miles city"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2177, generator_id='1', units=1)

p = "wy gen iii"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56596, generator_id='5', units=1)

### MONONGAHELA POWER COMPANY

In [180]:
p = "fort martin"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3943, generator_id='1', units=2)

p = "harrison"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3944, generator_id='1', units=3)

### Madison Gas and Electric Company

In [181]:
p = "plant name: badger hollow 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64393, generator_id='GEN2', units=1)

p = "plant name: badger hollow solar park"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62955, generator_id='GEN1', units=1)

p = "plant name: bount station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3992, generator_id='6', units=2)

p = "plant name: columbia total"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8023, generator_id='1', units=2)

p = "plant name: columbia 1" # part of prior total
df = delete_plant_rows(plant_names=[p])

p = "plant name: columbia 2" # part of prior total
df = delete_plant_rows(plant_names=[p])

p = "plant name: elm road 1"
df = delete_plant_rows(plant_names=[p])

p = "plant name: elm road 2"
df = delete_plant_rows(plant_names=[p])

p = "plant name: elm road total"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56068, generator_id='1', units=2)

p = "plant name: fitchburg-2 units"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3991, generator_id='1', units=2)

p = "plant name: forward energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56942, generator_id='1', units=2)

p = "plant name: m34/marinette"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4076, generator_id='31', units=4)

p = "plant name: nine springs"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=9674, generator_id='GT1', units=1)

p = "plant name: red barn wind"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65282, generator_id='WT', units=1)

p = "plant name: saratoga wind farm"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61070, generator_id='SWE', units=1)

p = "plant name: sycamore-2 units"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3993, generator_id='1', units=2)

p = "plant name: top of iowa 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56386, generator_id='TOI3', units=1)

p = "plant name: two creeks solar park"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=63105, generator_id='1', units=1)

p = "plant name: west campus"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7991, generator_id='1', units=3)

p = "plant name: west riverside"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64020, generator_id='CTG3', units=4)

### MidAmerican Energy Company

In [182]:
p = "coralville"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1079, generator_id='1', units=4)

p = "electrifarm"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6063, generator_id='1', units=3)

p = "greater des moines energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7985, generator_id='GT1', units=3)

p = "louisa"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6664, generator_id='1', units=1)

p = "merl parr"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1097, generator_id='1', units=2)

p = "moline"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=899, generator_id='GT1', units=8)

p = "neal #3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1091, generator_id='3', units=1)

p = "neal #4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7343, generator_id='4', units=1)

p = "ottumwa"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6254, generator_id='1', units=1)

p = "pleasant hill"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7145, generator_id='1', units=3)

p = "quad-cities"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=880, generator_id='1', units=2)

p = "river hills"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1084, generator_id='1', units=8)

p = "sycamore"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8029, generator_id='1', units=2)
df = change_plant_name(plant_name_ferc1=p, utility_name="MidAmerican Energy Company", new_name="sycamore_midAm") 

p = "walter scott #3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1082, generator_id='3', units=2)

p = "walter scott #4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1082, generator_id='4', units=2)

### Midwest Energy Inc.

In [183]:
p = "bird city"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1224, generator_id='1', units=2)

p = "colby gt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1225, generator_id='GT1', units=1)

p = "goodman energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56497, generator_id='1', units=12)

p = "great bend"
df = is_retired(plant_name_ferc1=p)
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1334, generator_id='1', units=6)

### Mississippi Power Company

In [184]:
p = "chevron"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2047, generator_id='1', units=5)

p = "daniel - combined cycle"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6073, generator_id='3', units=8)

p = "daniel - steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6073, generator_id='1', units=8)

p = "eaton"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2046, generator_id='1', units=3)
df = is_retired(plant_name_ferc1=p)

p = "ratcliffe"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=57037, generator_id='1A', units=3)

p = "sweat - ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2048, generator_id='A', units=3)

p = "sweat - steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2048, generator_id='1', units=3)
df = is_retired(plant_name_ferc1=p)

p = "watson - steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2049, generator_id='4', units=6)

p = "watson ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2049, generator_id='A', units=6)

### National Grid Generation LLC

In [185]:
p = "east hampton gt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2512, generator_id='1', units=4)

p = "ef barrett gt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2511, generator_id='GT1', units=14)

p = "ef barrett steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2511, generator_id='ST1', units=14)

p = "far rockaway steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2513, generator_id='4', units=1)
df = is_retired(plant_name_ferc1=p)

p = "glenwood gt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7869, generator_id='GT4', units=3)

p = "glenwood steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2514, generator_id='4', units=4)
df = is_retired(plant_name_ferc1=p)

p = "holtsville gt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8007, generator_id='1', units=10)

p = "northport gt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2516, generator_id='GT1', units=5)

p = "northport steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2516, generator_id='ST1', units=5)

p = "port jefferson gt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2517, generator_id='GT1', units=7)

p = "port jefferson steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2517, generator_id='3', units=7)

p = "shoreham gt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2518, generator_id='GT1', units=2)

p = "south hampton gt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2519, generator_id='1', units=1)

p = "southhold gt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2520, generator_id='1', units=1)

p = "wading river gt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7146, generator_id='1', units=3)

p = "west babylon gt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2521, generator_id='4', units=1)
df = add_notes(plant_name_ferc1=p, notes="Mothballed")
df = is_retired(plant_name_ferc1=p)


### Nevada Power Company, d/b/a NV Energy

In [186]:
p = "clark 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2322, generator_id='GT4', units=21)

p = "clark 5,6,7,8,9,10"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2322, generator_id='GT5', units=21)

p = "clark peakers 11-22"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2322, generator_id='11', units=21)

p = "harry allen 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7082, generator_id='GT1', units=5)

p = "harry allen 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7082, generator_id='GT4', units=5)

p = "harry allen 5,6,7"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7082, generator_id='5', units=5)

p = "higgins"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55687, generator_id='A01', units=3)

p = "lenzie 1 & 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55322, generator_id='CTG1', units=6)

p = "lv generation"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=10761, generator_id='GEN1', units=8)

p = "silverhawk"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55841, generator_id='CT1', units=3)

p = "sun peak 3, 4, 5"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=54854, generator_id='5657', units=3)

### NorthWestern Corporation

In [187]:
p = "aberdeen #1"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3338, generator_id='GT1', units=2)

p = "aberdeen #2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3338, generator_id='2', units=2)

p = "beethoven wind"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59187, generator_id='B&H80', units=1)

p = "bob glanzer"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65295, generator_id='HG1', units=6)

p = "colstrip 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6076, generator_id='4', units=2)

p = "dggs"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=67766, generator_id='YGS01', units=18)

p = "neal"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1091, generator_id='3', units=3)

p = "spion kop"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=58218, generator_id='SKW25', units=1)

p = "two dot"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59003, generator_id='1', units=1)

p = "yankton"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8034, generator_id='1', units=4)


### Northern Indiana Public Service Company

In [188]:
p = "michigan city (steam)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=997, generator_id='12', units=3)

p = "rm schahfer (combustion turbine)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6085, generator_id='16A', units=6)

p = "rm schahfer (steam)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6085, generator_id='17', units=6)

p = "sugar creek (combine cycle)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55364, generator_id='CT01', units=6)

p = "sugar creek (steam)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55364, generator_id='ST1', units=6)
df.loc[df["plant_name_ferc1"] == p, "capacity_mw"] = 213

### Northern States Power Company (Minnesota)

In [189]:
p = "a s king"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1915, generator_id='1', units=1)

p = "angus anson"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7237, generator_id='1', units=3)

p = "black dog 2, 5, & 6"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1904, generator_id='2', units=9)

p = "blue lake"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8027, generator_id='7', units=6)

p = "high bridge 7,8,9"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1912, generator_id='7', units=5)

p = "inver hills"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1913, generator_id='1', units=8)

p = "monticello"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1922, generator_id='1', units=1)

p = "prairie island"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1925, generator_id='1', units=2)

p = "riverside"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1927, generator_id='10', units=4)
df = change_plant_name(plant_name_ferc1=p, utility_name="Northern States Power Company (Minnesota)", new_name="riverside (NSP)") 

p = "sherburne county"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6090, generator_id='1', units=3)

p = "wilmarth"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1934, generator_id='1', units=2)

### Northern States Power Company (Wisconsin)

In [190]:
p = "bay front"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3982, generator_id='6', units=3)

p = "french island 1 & 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4005, generator_id='1', units=4)

p = "french island 3&4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4005, generator_id='3', units=4)

p = "wheaton"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4014, generator_id='1', units=6)

### Ohio Valley Electric Corporation

In [191]:
p = "kyger creek"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2876, generator_id='1', units=5)

### Oklahoma Gas and Electric Company

In [192]:
p = "frontier"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=50558, generator_id='GT01', units=2)

p = "horseshoe lake"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2951, generator_id='10', units=8)

p = "river valley"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=10671, generator_id='GEN1', units=2)

p = "mcclain"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55457, generator_id='CT1', units=3)

p = "muskogee"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2952, generator_id='4', units=4)

p = "mustang cts"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2953, generator_id='GT1', units=12)

p = "redbud"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55463, generator_id='CT01', units=8)

p = "seminole"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2956, generator_id='1', units=4)

p = "sooner"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6095, generator_id='1', units=3)

p = "tinker"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=63628, generator_id='5A', units=4)

### Old Dominion Electric Cooperative

In [193]:
p = "clover"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7213, generator_id='1', units=2)

p = "louisa"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7837, generator_id='1', units=5)

p = "marsh run"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7836, generator_id='1', units=4)

p = "north anna"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6168, generator_id='1', units=3)

p = "wildcat point"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59220, generator_id='CT1', units=3)


### Otter Tail Power Company

In [194]:
p = "astoria"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61144, generator_id='1', units=1)

p = "jamestown"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2801, generator_id='1', units=2)

p = "lake preston"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3352, generator_id='1A', units=2)

p = "solway"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7947, generator_id='1', units=2)

### PACIFIC GAS AND ELECTRIC COMPANY

In [195]:
p = "colusa generating station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56532, generator_id='A', units=3)

p = "diablo canyon 1 & 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6099, generator_id='1', units=2)

p = "gateway gen station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56476, generator_id='A', units=3)

p = "humboldt gen station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=246, generator_id='IC1', units=12)


### PacifiCorp

In [196]:
p = "blundell"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=299, generator_id='1', units=3)

p = "chehalis"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55662, generator_id='CA', units=3)

p = "colstrip"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6076, generator_id='3', units=4)

p = "craig"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6021, generator_id='1', units=3)

p = "currant creek"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56102, generator_id='CT1A', units=3)

p = "dave johnston"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4158, generator_id='1', units=4)

p = "gadsby peakers"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3648, generator_id='4', units=6)

p = "gadsby steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3648, generator_id='1', units=6)

p = "hayden"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=525, generator_id='1', units=2)

p = "hermiston"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=54761, generator_id='GEN1', units=4)

df = delete_plant_rows(plant_names=['hunter unit no. 1','hunter unit no. 2','hunter unit no. 3'])
p = "hunter - total plant"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6165, generator_id='1', units=3)

p = "huntington"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8069, generator_id='1', units=2)

p = "jim bridger"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8066, generator_id='1', units=4)

p = "lake side"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56237, generator_id='CT01', units=6)

p = "lake side 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56237, generator_id='CT02', units=6)

p = "naughton"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4162, generator_id='1', units=3)

p = "wyodak"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6101, generator_id='1', units=1)

### Portland General Electric Company

In [197]:
p = "beaver"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8073, generator_id='1', units=8)

p = "carty"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=58503, generator_id='GEN1', units=2)

p = "coyote springs"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7350, generator_id='1', units=2)

p = "port westward 1"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56227, generator_id='1', units=2)

p = "port westward 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=58266, generator_id='1', units=12)

### Public Service Company of Colorado

In [198]:
p = "alamosa"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=464, generator_id='CT1', units=2)

p = "blue spruce"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55645, generator_id='CT01', units=2)

p = "cherokee 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=469, generator_id='4', units=7)

p = "cherokee 5, 6, & 7"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=469, generator_id='5', units=7)

p = "comanche"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=470, generator_id='1', units=3)
df = change_plant_name(plant_name_ferc1=p, utility_name="Public Service Company of Colorado", new_name="comanche PSCo") 

p = "fort lupton"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8067, generator_id='1', units=2)

p = "fort st. vrain 1-4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6112, generator_id='1', units=6)

p = "fort st. vrain 5-6"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6112, generator_id='5', units=6)

p = "fruita"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=471, generator_id='1', units=1)

p = "manchief"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55127, generator_id='UN1', units=2)

p = "pawnee"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6248, generator_id='1', units=1)

p = "rocky mountain"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55835, generator_id='CTG1', units=3)

p = "valmont 5"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=477, generator_id='5', units=2)
df = is_retired(plant_name_ferc1=p)

p = "valmont 6, 7, & 8"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55207, generator_id='UN7', units=2)

### Public Service Company of New Mexico

In [199]:
p = "afton turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55210, generator_id='1', units=3)

p = "four corners (1)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2442, generator_id='1', units=5)

p = "la luz"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=58284, generator_id='1', units=2)

p = "lordsburg turbine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7967, generator_id='GT1', units=2)

p = "luna"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55343, generator_id='CTG1', units=3)

p = "palo verde (1)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6008, generator_id='1', units=3)

p = "reeves"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2450, generator_id='1', units=3)

p = "rio bravo"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55039, generator_id='GT1', units=1)

### Public Service Company of Oklahoma

In [200]:
p = "comanche"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8059, generator_id='1G1', units=4)

p = "north central wind"
df = add_notes(plant_name_ferc1=p, notes="North Central Wind Project appears to include multiple wind plants as listed in EIA, perhaps including Sundance and/or Maverick")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=63489, generator_id='GEN1', units=1)

p = "northeastern 1&2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2963, generator_id='1', units=7)
df = add_notes(plant_name_ferc1=p, notes="Listed in FERC1 as 'STEAM' but appears to be NGCC in EIA")

p = "northeastern 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2963, generator_id='3', units=7)

p = "riverside 1 & 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4940, generator_id='1', units=5)

p = "riverside 3 & 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4940, generator_id='3', units=5)

p = "rock falls"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61261, generator_id='RF1', units=1)

p = "southwestern 1 - 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2964, generator_id='1', units=6)

p = "southwestern 4 & 5"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2964, generator_id='4', units=6)

p = "tulsa"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2965, generator_id='2', units=4)

p = "weleetka"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2966, generator_id='4', units=4)


### Puget Sound Energy, Inc.

In [201]:
p = "colstrip 3 & 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6076, generator_id='1', units=4)

p = "encogen"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7870, generator_id='CTG1', units=4)

p = "ferndale"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=54537, generator_id='CT1A', units=3)

p = "frederickson"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=99, generator_id='1', units=2)

p = "frederickson 1"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55818, generator_id='FICT', units=2)

p = "fredonia 1&2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=607, generator_id='1', units=4)

p = "fredonia 3&4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=607, generator_id='3', units=4)

p = "goldendale"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55482, generator_id='G1', units=2)

p = "hopkins ridge"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56255, generator_id='1', units=2)

p = "lower snake river"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=57195, generator_id='LSR 1', units=1)

p = "mint farm"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55700, generator_id='1STG', units=2)

p = "sumas"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=54476, generator_id='GEN1', units=2)

p = "whitehorn"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6120, generator_id='2', units=2)

p = "wild horse"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56322, generator_id='1', units=3)

### San Diego Gas & Electric Company

In [202]:
p = "cuyamaca"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55512, generator_id='CPP6', units=1)

p = "desert star"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55077, generator_id='ED01', units=3)

p = "miramar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56232, generator_id='1', units=2)

p = "palomar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55985, generator_id='CTG1', units=3)

### Sierra Pacific Power Company d/b/a NV Energy

In [203]:
p = "clark mountain 3 & 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2336, generator_id='GT3', units=12)

p = "ft churchill 1 & 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2330, generator_id='1', units=2)

p = "tracy 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2336, generator_id='3', units=12)

p = "tracy 4&5-pinon pine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2336, generator_id='4', units=12)

p = "tracy 8 - 10"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2336, generator_id='8', units=12)

p = "valmy 1 & 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8224, generator_id='1', units=2)

### South Carolina Electric & Gas Company

In [204]:
p = "canadys"
df = is_retired(plant_name_ferc1=p)
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3280, generator_id='1', units=3)

p = "coit #1 peaking unit"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3281, generator_id='1', units=2)

p = "coit #2 peaking unit"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3281, generator_id='2', units=2)

p = "coit combined"
df = delete_plant_rows(plant_names=[p])

p = "columbia energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55386, generator_id='CT1', units=3)

p = "cope"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7210, generator_id='ST1', units=1)

p = "hagood #4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3285, generator_id='4', units=3)

p = "hagood #5"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3285, generator_id='5', units=3)

p = "hagood #6"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3285, generator_id='6', units=3)

p = "hagood combined"
df = delete_plant_rows(plant_names=[p])

p = "hardeeville peaking"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3286, generator_id='1', units=1)
df = is_retired(plant_name_ferc1=p)

p = "jasper"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55927, generator_id='CT1', units=4)

p = "major maintenance accrual"
df = delete_plant_rows(plant_names=[p])

p = "mcmeekin"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3287, generator_id='1', units=2)

p = "parr #1 & #2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3291, generator_id='GT1', units=6)
df = is_retired(plant_name_ferc1=p)

p = "parr #3 & #4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3291, generator_id='GT3', units=6)
df = is_retired(plant_name_ferc1=p)

p = "parr combined"
df = delete_plant_rows(plant_names=[p])

p = "urquhart"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3295, generator_id='3', units=9)

p = "urquhart #1 peaking"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3295, generator_id='GT1', units=9)

p = "urquhart #2 peaking"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3295, generator_id='GT2', units=9)

p = "urquhart #3 peaking"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3295, generator_id='GT3', units=9)

p = "urquhart #4 peaking"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3295, generator_id='GT4', units=9)

p = "urquhart combined 1-4"
df = delete_plant_rows(plant_names=[p])

p = "urquhart combined cycle"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3295, generator_id='1', units=9)

p = "v.c. summer (2/3rds)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6127, generator_id='1', units=3)

p = "wateree"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3297, generator_id='1', units=2)

p = "williams #1 peaking"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3298, generator_id='1', units=3)
df = is_retired(plant_name_ferc1=p)

p = "williams #2 peaking"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3298, generator_id='2', units=3)
df = is_retired(plant_name_ferc1=p)

p = "williams combined"
df = delete_plant_rows(plant_names=[p])

p = "williams"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3298, generator_id='ST1', units=3)


### Southern California Edison Company

In [205]:
p = "barre peaker"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56474, generator_id='1', units=1)

p = "center peaker"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56475, generator_id='1', units=2)

p = "grapeland peaker"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56472, generator_id='1', units=2)

p = "mcgrath peaker"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56471, generator_id='1', units=1)

p = "mira loma peaker"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56473, generator_id='1', units=1)

p = "mountainview 3 & 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=358, generator_id='MV3A', units=8)

p = "offsite storage pipelines"
df = delete_plant_rows(plant_names=[p])

### Southern Indiana Gas and Electric Company

In [206]:
p = "a.b. brown station"
df = is_retired(plant_name_ferc1=p)
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6137, generator_id='1', units=6)

p = "a.b. brown turbine 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6137, generator_id='5', units=6)

p = "ab brown turbine 4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6137, generator_id='4', units=6)

p = "f.b. culley station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1012, generator_id='2', units=3)

p = "warrick unit #4"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6705, generator_id='4', units=4)


### Southwestern Electric Power Company

In [207]:
p = "*dolet hills (3)"
df = is_retired(plant_name_ferc1=p)
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=51, generator_id='1', units=1)

p = "*flint creek (1)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6138, generator_id='1', units=1)

p = "*pirkey (2)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7902, generator_id='1', units=1)
df = is_retired(plant_name_ferc1=p)

p = "arsenal hill"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1416, generator_id='5', units=1)

p = "harry d mattison"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56328, generator_id='1', units=4)

p = "knox lee"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3476, generator_id='5', units=4)

p = "lieberman"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1417, generator_id='3', units=4)

p = "turk (4)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56564, generator_id='1', units=1)

p = "welsh"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6139, generator_id='1', units=3)

p = "wilkes"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3478, generator_id='1', units=3)

### Southwestern Public Service Company

In [208]:
p = "carlsbad"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2453, generator_id='5', units=1)
df = is_retired(plant_name_ferc1=p)

p = "cunningham gas"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2454, generator_id='3', units=4)

p = "cunningham steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2454, generator_id='1', units=4)

p = "harrington"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6193, generator_id='1', units=3)

p = "jones station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3482, generator_id='1', units=4)

p = "jones station gas"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3482, generator_id='3', units=4)

p = "maddox gas"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2446, generator_id='2', units=3)

p = "maddox steam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2446, generator_id='1', units=3)

p = "moore county"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3483, generator_id='3', units=1)
df = is_retired(plant_name_ferc1=p)

p = "nichols station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3484, generator_id='1', units=3)

p = "plant x"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3485, generator_id='1', units=4)

p = "quay county"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=58125, generator_id='1', units=1)

p = "tolk"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6194, generator_id='1', units=2)


### System Energy Resources, Inc.

In [209]:
p = "grand gulf"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6072, generator_id='1', units=1)

### Tampa Electric Company

In [210]:
p = 'a'
df = add_notes(plant_name_ferc1=p, notes="EIA data shows half the capacity reported in FERC is retired")
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=645, generator_id='ST4', units=3)

### The Empire District Electric Company

In [211]:
p = "energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6223, generator_id='1', units=4)

p = "iatan (1&2)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6065, generator_id='1', units=2)

p = "plum point"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56456, generator_id='STG1', units=1)

p = "riverton (10-11-12)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1239, generator_id='10', units=7)

p = "stateline combined cycle"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7296, generator_id='2-1', units=4)

p = "stateline ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7296, generator_id='1', units=4)

### Tucson Electric Power Company

In [212]:
p = "demoss petrie"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=124, generator_id='GT2', units=1)

p = "four corners"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2442, generator_id='4', units=5)

p = "gila river"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59338, generator_id='CTG1', units=6)

p = "luna"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55343, generator_id='CTG1', units=3)

p = "north loop"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6088, generator_id='1', units=4)

p = "springerville"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8223, generator_id='1', units=8)

p = "sundt"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=126, generator_id='4', units=16)

p = "sundt (gas)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=126, generator_id='GT1', units=16)

p = "sundt (rice)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=126, generator_id='RIC1', units=16)

### UNION ELECTRIC COMPANY

In [213]:
p = "audrain ctg"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55234, generator_id='GT1', units=8)

p = "callaway"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6153, generator_id='1', units=1)

p = "fairgrounds ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2082, generator_id='1', units=1)

p = "goose creek ctg"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55496, generator_id='CT01', units=6)

p = "kinmundy"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55204, generator_id='1', units=2)

p = "kirksville ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2083, generator_id='1', units=1)
df = is_retired(plant_name_ferc1=p)

p = "labadie"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2103, generator_id='1', units=4)

p = "maryland heights lf"

p = "meramec"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2104, generator_id='3', units=6)
df = is_retired(plant_name_ferc1=p)

p = "meramec ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2104, generator_id='GT1', units=6)
df = is_retired(plant_name_ferc1=p)

p = "mexico ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6650, generator_id='1', units=1)

p = "moberly ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6651, generator_id='1', units=1)

p = "moreau ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6652, generator_id='1', units=1)

p = "peno creek ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7964, generator_id='GT1', units=4)

p = "pickneyville"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55202, generator_id='1', units=8)

p = "raccoon creek ctg"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55417, generator_id='CT01', units=4)

p = "rush island"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6155, generator_id='1', units=2)

p = "sioux"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2107, generator_id='1', units=2)

p = "venice ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=913, generator_id='GT2', units=11)

### UNS Electric, Inc.

In [214]:
p = "black mountain"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56482, generator_id='1', units=2)

p = "gila river"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59338, generator_id='CTG1', units=3)

p = "valencia"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6515, generator_id='GT1', units=4)

### Upper Peninsula Power Company

In [215]:
p = "gladstone"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7119, generator_id='1', units=1)

### VIRGINIA ELECTRIC AND POWER COMPANY

In [216]:
p = "altavista"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=10773, generator_id='1', units=1)

p = "bear garden"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56807, generator_id='1A', units=3)

p = "bedford"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65323, generator_id='BEDF', units=1)

p = "belcher"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60319, generator_id='1', units=1)

p = "brunswick"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=58260, generator_id='CT01', units=4)

p = "chesapeake"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3803, generator_id='6', units=12)

p = "chesterfield com cyc"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3797, generator_id='CT7', units=4)

p = "chestnut"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61011, generator_id='PV1', units=1)

p = "clover"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7213, generator_id='1', units=2)

p = "colonial trail west"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61985, generator_id='CTWS', units=1)

p = "costal virginia offshore wind"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59693, generator_id='OSW1', units=1)

p = "darbytown"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7212, generator_id='1', units=4)

p = "elizabeth river ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=52087, generator_id='GEN1', units=3)

p = "fort powhatan"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65322, generator_id='FTPW', units=1)

p = "gloucester solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=63031, generator_id='GLSO', units=1)

p = "gordonsville"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=54844, generator_id='GOR1', units=4)

p = "grassfield solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64135, generator_id='GFSO', units=1)

p = "grasshopper"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62813, generator_id='GRHS', units=1)

p = "gravel neck"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7032, generator_id='1', units=6)

p = "greensville"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59913, generator_id='CT01', units=4)

p = "gutenberg solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=63076, generator_id='GUTN', units=1)

p = "hollyfield solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61023, generator_id='1', units=1)

p = "ladysmith"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7839, generator_id='1', units=5)

p = "low moor"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3799, generator_id='GT1', units=4)

p = "maplewood solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65319, generator_id='MAPL', units=1)

p = "montross solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64093, generator_id='MOSO', units=1)

p = "morgans corner"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60422, generator_id='1', units=1)

p = "mount storm"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3954, generator_id='1', units=4)

p = "mount storm ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3954, generator_id='JF1', units=1)

p = "norge"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64086, generator_id='NORG', units=1)

p = "north anna"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6168, generator_id='1', units=3)

p = "northern neck"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3800, generator_id='GT1', units=4)

p = "oceana solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60584, generator_id='1', units=1)

p = "pecan solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60030, generator_id='PECAN', units=1)

p = "piney creek"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62768, generator_id='PCSOL', units=1)

p = "polyester"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=10771, generator_id='1', units=1)

p = "possum point com cyc"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3804, generator_id='6A', units=14)

p = "possum point ct"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3804, generator_id='GT1', units=14)

p = "puller solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62140, generator_id='PULL', units=1)

p = "pumpkinseed solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=63745, generator_id='GVSO', units=1)

p = "remington"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7838, generator_id='1', units=4)

p = "remington solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59685, generator_id='1', units=1)

p = "rochambeau"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65321, generator_id='RBSO', units=1)

p = "rosemary"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=50555, generator_id='GEN1', units=3)

p = "sadler"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62814, generator_id='SADL', units=1)

p = "scott timber solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60968, generator_id='PV1', units=1)

p = "solidago"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=66999, generator_id='SLDO', units=1)

p = "southampton"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=10774, generator_id='1', units=1)

p = "spring grove 1"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61986, generator_id='SGIS', units=1)

p = "surry"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=3806, generator_id='1', units=1)

p = "sycamore"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64136, generator_id='SYSO', units=1)

p = "virginia city"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56808, generator_id='1', units=1)

p = "warren"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55939, generator_id='CT01', units=4)

p = "whitehouse solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60319, generator_id='1', units=1)

p = "winterberry"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=63031, generator_id='GLSO', units=1)

p = "woodland solar"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60318, generator_id='1', units=1)

### Wabash Valley Power Association, Inc.

In [217]:
p = "gibson unit 5"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6113, generator_id='5', units=5)

p = "holland"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55334, generator_id='CTG1', units=3)

p = "prairie state"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55856, generator_id='PC1', units=2)

### Westar Energy, Inc.

In [218]:
p = "central plains"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56818, generator_id='1', units=1)

p = "emporia ctf"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56502, generator_id='1', units=7)

p = "flat ridge"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56819, generator_id='1', units=1)

p = "gordan evans ctf"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1240, generator_id='GT1', units=6)

p = "hutchinson"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1248, generator_id='GT1', units=9)

p = "hutchinson w/diesel"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1248, generator_id='GT4', units=9)

p = "jeffrey (jec)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6068, generator_id='1', units=3)

p = "lawrence"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1250, generator_id='4', units=4)

p = "persimmon creek"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61876, generator_id='PCWF1', units=1)

p = "spring creek"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55651, generator_id='CT01', units=4)

p = "western plains"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60689, generator_id='1', units=1)

### Westar Generating, Inc.

In [219]:
p = "state line"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7296, generator_id='1', units=4)

### Wisconsin Electric Power Company

In [220]:
p = "badger hollow ii"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64393, generator_id='GEN2', units=1)

p = "blue sky green field"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56391, generator_id='1', units=1)

p = "concord"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7159, generator_id='1', units=4)

p = "elm road"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56068, generator_id='1', units=2)

p = "germantown"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6253, generator_id='1', units=5)

p = "glacier hills"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=57199, generator_id='1', units=1)

p = "montfort"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55742, generator_id='ER15', units=1)

p = "paris"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7270, generator_id='1', units=4)

p = "port washington"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4040, generator_id='1CT1', units=11)

p = "rothschild"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=58124, generator_id='1', units=1)

p = "south oak creek"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4041, generator_id='5', units=5)

p = "valley"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4042, generator_id='1', units=1)

p = "west riverside"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64020, generator_id='CTG3', units=4)

p = "weston rice"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=66059, generator_id='W11', units=7)

p = "whitewater"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55011, generator_id='CTG1', units=2)

### Wisconsin Power and Light Company

In [221]:
p = "albany"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64997, generator_id='PV1', units=1)

p = "bear creek"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65009, generator_id='PV1', units=1)

p = "beaver dam"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65001, generator_id='PV1', units=1)

p = "bent tree"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=57198, generator_id='1', units=1)

p = "cassville"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64995, generator_id='PV1', units=1)

p = "cedar ridge"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56347, generator_id='1', units=1)

p = "columbia"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8023, generator_id='1', units=2)

p = "crawfish river"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65012, generator_id='PV1', units=1)

p = "edgewater 5"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4050, generator_id='5', units=3)

p = "forward wind"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56942, generator_id='1', units=2)

p = "kossuth"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62103, generator_id='1', units=1)

p = "neenah"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55135, generator_id='CT01', units=1)

p = "north rock"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65010, generator_id='PV1', units=1)

p = "onion river"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65008, generator_id='PV1', units=1)

p = "paddock"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64998, generator_id='PV1', units=1)

p = "riverside"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55641, generator_id='CTG1', units=3)

p = "s fond du lac"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7203, generator_id='CT1', units=4)

p = "sheboygan falls"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56166, generator_id='1', units=2)

p = "springfield"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64996, generator_id='PV1', units=1)

p = "wautoma"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65000, generator_id='PV1', units=1)

p = "west riverside"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=64020, generator_id='CTG3', units=4)

p = "wood county"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65011, generator_id='PV1', units=1)

### Wisconsin Public Service Corporation

In [222]:
p = "badger hollow i"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=62955, generator_id='GEN1', units=1)

p = "crane creek"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56831, generator_id='1', units=1)

p = "de pere energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55029, generator_id='CT01', units=1)

p = "forward wind"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56942, generator_id='1', units=2)

p = "fox energy center"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56031, generator_id='CTG1', units=3)

p = "pulliam 31"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4072, generator_id='31', units=9)

p = "red barn wind park"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=65282, generator_id='WT', units=1)

p = "two creeks"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=63105, generator_id='1', units=1)

p = "west marinette"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4076, generator_id='31', units=3)

p = "weston"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4078, generator_id='3', units=6)

p = "weston rice"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=66059, generator_id='W11', units=7)

p = "weston w31, w32"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=4078, generator_id='31', units=6)
df = is_retired(plant_name_ferc1=p)

p = "whitewater"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55011, generator_id='CTG1', units=2)

### Wolverine Power Supply Cooperative, Inc.

In [223]:
p = "alpine"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=59926, generator_id='AI1', units=2)

p = "burnips"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1880, generator_id='8', units=3)

p = "campbell #3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1710, generator_id='1', units=4)

p = "gaylord"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7932, generator_id='1', units=3)

p = "hersey"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1877, generator_id='10', units=11)

p = "sumpter"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=7972, generator_id='1', units=4)

p = "tower"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1873, generator_id='GT4', units=4)

p = "vestaburg"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1881, generator_id='8', units=6)

p = "vestaburg-fo"
df = delete_plant_rows(plant_names=[p])

### basin electric power cooperative (pudl determined)

In [224]:
p = "antelope valley station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6469, generator_id='1', units=2)

p = "culbertson station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56606, generator_id='1', units=1)

p = "deer creek station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56610, generator_id='1', units=2)

p = "dry fork station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56609, generator_id='1', units=1)

p = "groton generation station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=56238, generator_id='GT01', units=2)

p = "laramie river station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6204, generator_id='1', units=3)

p = "leland olds station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2817, generator_id='1', units=2)

p = "lonesome creek station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=57943, generator_id='1', units=6)

p = "pioneer generation station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=57881, generator_id='1', units=23)

p = "spirit mound station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6092, generator_id='1', units=2)

p = "wisdom generating station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1217, generator_id='1', units=1)

p = "wyoming distributed generation"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8028, generator_id='1', units=3)

### entergy mississippi, llc

In [225]:
p = "attala"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55220, generator_id='A01', units=3)

p = "baxter wilson"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=2050, generator_id='1', units=2)
df = is_retired(plant_name_ferc1=p)

p = "choctaw"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55706, generator_id='CTG1', units=4)

p = "gerald andrus"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=8054, generator_id='1', units=1)

p = "hinds"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=55218, generator_id='H01', units=4)

p = "independence"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6641, generator_id='1', units=2)


### entergy new orleans, llc

In [226]:
p = "new orleans power station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=60928, generator_id='RICE1', units=7)

### evergy kansas south, inc.

In [227]:
p = "gordon evans w/diesel"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1240, generator_id='5', units=6)

p = "jeffrey 20%"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6068, generator_id='1', units=3)

p = "lacygne 1 (50%)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1241, generator_id='1', units=2)

p = "lacygne 2 (50%)"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=1241, generator_id='1', units=2)

p = "wolf creek 47%"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=210, generator_id='1', units=1)

### tri-state generation and transmission association (pudl determined)

In [228]:
p = "craig station unit 3"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6021, generator_id='3', units=3)

p = "craig station units 1 & 2"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6021, generator_id='1', units=3)

p = "j.m. shafer generating station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=50707, generator_id='LMA', units=7)

p = "laramie river station"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=6204, generator_id='1', units=3)

### upper michigan energy resources company (pudl determined)

In [229]:
p = "kuester"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61392, generator_id='K1', units=7)

p = "mihm"
df = add_eia_info2(plant_name_ferc1=p, plant_id_eia=61391, generator_id='M1', units=3)

In [230]:
df.to_csv(os.path.join(root, 'data', 'ferc1_plants_utilitynames.csv'), index=False)

In [231]:
#gen[gen['plant_name_eia'] == "Greenwood"]